# Procesamiento por Lotes (Batch) y en Tiempo Real (Streaming) + Data Mesh

Objetivos:
- Diferenciar batch vs streaming y su impacto arquitectónico (Lambda/Kappa).
- Entender principios de Data Mesh para escalar y descentralizar dominios.
- Diseñar flujos de datos, orquestación, y gestionar latencia/eventos.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Almacenamiento e Integración de Datos (DW, Data Lake, Lakehouse, ETL/ELT)](01_almacenamiento_e_integracion.ipynb) | Siguiente: [Arquitecturas Lambda y Kappa + Event-Driven & SLOs](03_lambda_kappa_event_driven.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Batch vs Streaming
2. 2. Impacto Arquitectónico: Lambda vs Kappa
3. 3. Data Mesh: Principios y Práctica
4. 4. Gestión de Eventos y Latencia
5. 5. Orquestación y Operación
6. 6. Ecosistema de Herramientas (ejemplos)
7. 7. Ejercicio Práctico
8. 8. Resumen

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Procesamiento por Lotes (Batch) y en Tiempo Real (Streaming) + Data Mesh"] --> C["Notebook siguiente"]
```


## 1. Batch vs Streaming

Batch: procesamiento en ventanas discretas (minutos/horas/días).
Streaming: procesamiento continuo de eventos a baja latencia (ms/segundos).

| Atributo | Batch | Streaming |
|---|---|---|
| Latencia | Alta (programada) | Baja (casi real-time) |
| Volumen | Alto por corrida | Flujo constante |
| Orquestación | Schedulers (Airflow/Prefect) | Runners/Jobs continuos (Flink/Spark) |
| Uso típico | Cierres diarios, BI | Detección fraude, monitoreo, alertas |
| Estado | Usualmente sin estado | Stateful (ventanas, watermarks) |

Conceptos clave en streaming: ventanas (tumbling/sliding), event time vs processing time, watermarks, backpressure, exactamente-una-vez (exactly-once).

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(0)
ts = pd.date_range('2025-01-01', periods=100, freq='min')
events = pd.DataFrame({'event_time': ts, 'valor': np.random.poisson(2.5, size=100)})

print('Batch - suma del día:', events['valor'].sum())

events_mb = events.copy()
events_mb['window_15'] = events_mb['event_time'].dt.floor('15min')
micro = events_mb.groupby('window_15', as_index=False)['valor'].sum()
print('\nMicro-batch 15min (primeras filas):')
print(micro.head())

state = 0
for i, row in events.iterrows():
    state += row['valor']
    if i % 30 == 0 and i > 0:
        print(f'Evento {i}: estado acumulado={state}')
print('Streaming - estado final:', state)
print('\nNota: streaming real implica ventanas por event_time y watermarks.')


## 2. Impacto Arquitectónico: Lambda vs Kappa

- Lambda: dos caminos (batch y speed/stream) reconciliados en serving layer.
- Kappa: un solo camino streaming; batch se simula reprocesando el log de eventos.

Ventajas/Desventajas:
- Lambda: flexible pero más compleja (dos pipelines).
- Kappa: simplicidad operativa si todos los casos caben en streaming.

In [ ]:
# Diagrama simple Lambda/Kappa
import matplotlib.pyplot as plt
import matplotlib.patches as p
fig, ax = plt.subplots(1, 2, figsize=(12,4))
for a in ax: a.axis('off')
# Lambda
ax[0].set_title('Lambda')
ax[0].add_patch(p.FancyBboxPatch((0.05,0.6),0.25,0.25,boxstyle='round',facecolor='#A0E7E5'))
ax[0].text(0.175,0.725,'Batch',ha='center')
ax[0].add_patch(p.FancyBboxPatch((0.05,0.15),0.25,0.25,boxstyle='round',facecolor='#B4F8C8'))
ax[0].text(0.175,0.275,'Speed',ha='center')
ax[0].add_patch(p.FancyBboxPatch((0.38,0.35),0.25,0.3,boxstyle='round',facecolor='#FBE7C6'))
ax[0].text(0.505,0.5,'Serving Layer',ha='center')
ax[0].annotate('', xy=(0.38,0.5), xytext=(0.3,0.725), arrowprops=dict(arrowstyle='->'))
ax[0].annotate('', xy=(0.38,0.5), xytext=(0.3,0.275), arrowprops=dict(arrowstyle='->'))
# Kappa
ax[1].set_title('Kappa')
ax[1].add_patch(p.FancyBboxPatch((0.05,0.35),0.25,0.3,boxstyle='round',facecolor='#B4F8C8'))
ax[1].text(0.175,0.5,'Streaming',ha='center')
ax[1].add_patch(p.FancyBboxPatch((0.38,0.35),0.25,0.3,boxstyle='round',facecolor='#FBE7C6'))
ax[1].text(0.505,0.5,'Serving Layer',ha='center')
ax[1].annotate('', xy=(0.38,0.5), xytext=(0.3,0.5), arrowprops=dict(arrowstyle='->'))
plt.tight_layout(); plt.show()

## 3. Data Mesh: Principios y Práctica

Data Mesh no es una herramienta, es un modelo operativo para escalar datos en organizaciones grandes. Principios:
1. Propiedad por dominio: equipos dueños de sus datos (domain ownership).
2. Datos como producto: contratos, calidad, discoverability y SLAs.
3. Plataforma self-serve: herramientas comunes (ingestión, catálogo, linaje, CI/CD).
4. Gobernanza federada: estándares compartidos + autonomía local.

Artefactos clave: contratos de datos (esquema, semántica, SLO de frescura), catálogos, versionado de esquemas, políticas de acceso.

Relación con batch/streaming: cada dominio publica productos de datos (batch o eventos) con acuerdos explícitos de latencia/calidad.

## 4. Gestión de Eventos y Latencia

- Semánticas de entrega: at-least-once, at-most-once, exactly-once.
- Idempotencia: consumidores deben tolerar reintentos/deduplicar.
- Versionado de eventos: compatibilidad backward/forward (Schema Registry).
- Watermarks: gestionan retrasos y eventos desordenados.
- Backpressure: control de flujo cuando consumidores están saturados.
- SLOs de latencia y frescura (p.ej., P95 < 5s).

## 5. Orquestación y Operación

- Batch: DAGs con dependencias, retries, alertas (Airflow, Prefect, Dagster).
- Streaming: jobs de larga ejecución; despliegue/observabilidad (Flink/Spark/Kafka Streams).
- Observabilidad: métricas (lag, throughput), logs estructurados, tracing, lineage.
- Calidad: expectativas automáticas, validaciones por capa (bronze→gold).
- Seguridad: ACLs por tópico/tabla, PII masking, cifrado en tránsito y reposo.

In [ ]:
# Pseudoejemplo de DAG (texto) para batch diario y stream de eventos
dag = '''
DAG: retail_daily_batch
  - extract_pos (06:00)
  - extract_crm (06:10)
  - transform_silver (06:30) depends [extract_pos, extract_crm]
  - build_gold_kpis (07:00) depends [transform_silver]

STREAM JOB: clicks_stream_processor
  - source: kafka topic=web.clicks
  - ops: parse_json -> window(tumbling=5m) -> agg -> sink=gold.click_kpi
  - metrics: lag, records/sec, errors
'''
print(dag)

## 6. Ecosistema de Herramientas (ejemplos)

- Mensajería/Eventos: Kafka, Pulsar, Kinesis, Pub/Sub, Event Hubs.
- Procesamiento: Flink, Spark Structured Streaming, Kafka Streams, Beam.
- CDC: Debezium, Fivetran/Hevo/Stitch (SaaS).
- Orquestación: Airflow, Prefect, Dagster.
- Catálogo/Gobernanza: Google Cloud Knowledge Catalog, Atlas, Collibra, Amundsen.
- Observabilidad/Calidad: OpenLineage, Great Expectations, Monte Carlo.

## 7. Ejercicio Práctico

Escenario: E-commerce con pagos, envíos y clics web.

Tareas:
1. Define SLOs de latencia por dominio (p.ej., fraude P95<3s; BI diario antes de 07:00).
2. Elige Lambda o Kappa y justifica.
3. Diseña tópicos/eventos por dominio (contratos y versiones).
4. Propón pipeline batch para KPIs diarios y stream para alertas.
5. Define métricas de observabilidad (lag, throughput, error rate) y alertas.

In [ ]:
# Plantilla de solución
plantilla = '''
# Solución (borrador)

1) SLOs de latencia: [especifica por dominio]
2) Patrón Lambda/Kappa: [elección y motivos]
3) Contratos de eventos: [tópicos, esquema, versión]
4) Pipelines: [batch diario y stream continuo]
5) Observabilidad: [métricas y umbrales de alerta]
'''
print(plantilla)

## 8. Resumen

- Batch y streaming atienden necesidades distintas; pueden coexistir.
- Lambda vs Kappa: elige según complejidad y requisitos de latencia.
- Data Mesh alinea equipos por dominio y transforma datos en productos.
- Gestionar latencia y eventos requiere contratos, idempotencia y observabilidad.

---
**Referencias**
- Kreps, J. I Heart Logs.
- Kleppmann, M. Designing Data-Intensive Applications.
- Chen, Z. Stream Processing with Apache Flink.
- Dehghani, Z. Data Mesh: Delivering Data-Driven Value at Scale.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Apache Kafka: https://kafka.apache.org/
- Referencias del curso: ../04-recursos/referencias.md
